<a href="https://colab.research.google.com/github/AllyApitchaya/msc-adr-prediction/blob/main/notebooks/06_multimodal_fusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ===== Cell 1: Setup and environment check =====
import os
import torch
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Select device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {device}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU")

# Project root path (adjust if your folder has a different name)
PROJECT = '/content/drive/MyDrive/MSc_ADR_Project'
print(f"Project path    : {PROJECT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PyTorch version : 2.10.0+cu128
Device          : cuda
GPU             : Tesla T4
Project path    : /content/drive/MyDrive/MSc_ADR_Project


In [3]:
# ===== Check files from previous notebooks =====
for folder in ['data', 'data/embeddings', 'models', 'results']:
    path = os.path.join(PROJECT, folder)
    print(f"\n{folder}/")
    if os.path.exists(path):
        for f in sorted(os.listdir(path)):
            print(f"   {f}")
    else:
        print("   (folder does not exist)")


data/
   embeddings
   processed
   raw

data/embeddings/
   adr_biobert_embeddings.npz

models/
   gnn_model.pt

results/
   baseline_xgboost_metrics.csv
   disproportionality_results.csv
   gnn_test_metrics.csv


In [4]:
# ===== Inspect previous-notebook artefacts =====
import os

# 1. What is inside data/processed/
proc = os.path.join(PROJECT, 'data/processed')
print("data/processed/")
for f in sorted(os.listdir(proc)):
    print(f"   {f}")

# 2. Inspect one CSV to see its columns
import pandas as pd
import glob

csv_files = glob.glob(os.path.join(proc, '*.csv'))
if csv_files:
    sample = csv_files[0]
    df = pd.read_csv(sample)
    print(f"\nSample CSV: {os.path.basename(sample)}")
    print(f"   shape   : {df.shape}")
    print(f"   columns : {list(df.columns)}")
    print(df.head(3))

# 3. Inspect the saved GNN checkpoint
ckpt = torch.load(os.path.join(PROJECT, 'models/gnn_model.pt'),
                  map_location='cpu')
print(f"\ngnn_model.pt type: {type(ckpt)}")
if isinstance(ckpt, dict):
    print("   keys:", list(ckpt.keys())[:10])

data/processed/
   drug_adr_with_structures.csv
   faers_2020_2022_diabetes_adr.csv
   faers_2020q1_diabetes_adr.csv
   test.csv
   train.csv
   val.csv

Sample CSV: faers_2020q1_diabetes_adr.csv
   shape   : (9754, 5)
   columns : ['primaryid', 'caseid', 'drugname', 'role_cod', 'pt']
   primaryid    caseid   drugname role_cod  \
0  100826462  10082646  METFORMIN       PS   
1  100826462  10082646  METFORMIN       PS   
2  102652853  10265285  METFORMIN       PS   

                                     pt  
0                      Cardiac disorder  
1                      Drug interaction  
2  Diabetes mellitus inadequate control  

gnn_model.pt type: <class 'collections.OrderedDict'>
   keys: ['gat1.att_src', 'gat1.att_dst', 'gat1.bias', 'gat1.lin.weight', 'gat2.att_src', 'gat2.att_dst', 'gat2.bias', 'gat2.lin.weight', 'adr_embed.weight', 'classifier.0.weight']


In [5]:
# ===== Inspect the actual training data =====
import pandas as pd

train = pd.read_csv(os.path.join(PROJECT, 'data/processed/train.csv'))
print("train.csv")
print(f"   shape   : {train.shape}")
print(f"   columns : {list(train.columns)}")
print(train.head(5))
print()

# Check the label column distribution (positive vs negative pairs)
print("dtypes:")
print(train.dtypes)

# Inspect the structures file (likely holds SMILES for the GNN)
struct = pd.read_csv(os.path.join(PROJECT, 'data/processed/drug_adr_with_structures.csv'))
print("\ndrug_adr_with_structures.csv")
print(f"   shape   : {struct.shape}")
print(f"   columns : {list(struct.columns)}")
print(struct.head(3))

train.csv
   shape   : (8240, 5)
   columns : ['drug', 'pubchem_cid', 'smiles', 'adr', 'label']
          drug  pubchem_cid  \
0    glipizide         3478   
1  nateglinide      5311309   
2    metformin         4091   
3    glyburide         3488   
4    glyburide         3488   

                                              smiles  \
0  CC1=CN=C(C=N1)C(=O)NCCC2=CC=C(C=C2)S(=O)(=O)NC...   
1         CC(C)C1CCC(CC1)C(=O)NC(CC2=CC=CC=C2)C(=O)O   
2                                  CN(C)C(=N)N=C(N)N   
3  COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...   
4  COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...   

                            adr  label  
0                  Constipation      1  
1  Hyperplastic cholecystopathy      0  
2         Hepatitis cholestatic      1  
3           Portal hypertension      0  
4                   Hypotension      1  

dtypes:
drug           object
pubchem_cid     int64
smiles         object
adr            object
label           int64
dtype: object



In [6]:
# ===== Cell 2: Load processed data and BioBERT embeddings =====
import numpy as np
import pandas as pd

# --- Load the three data splits ---
train_df = pd.read_csv(os.path.join(PROJECT, 'data/processed/train.csv'))
val_df   = pd.read_csv(os.path.join(PROJECT, 'data/processed/val.csv'))
test_df  = pd.read_csv(os.path.join(PROJECT, 'data/processed/test.csv'))

print("Data splits")
print(f"   train : {train_df.shape}")
print(f"   val   : {val_df.shape}")
print(f"   test  : {test_df.shape}")

# Label balance (how many positive vs negative pairs)
for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    pos = df['label'].sum()
    n   = len(df)
    print(f"   {name} positives: {pos}/{n} ({pos/n:.1%})")

# --- Load the cached BioBERT embeddings from notebook 05 ---
emb_data = np.load(os.path.join(PROJECT, 'data/embeddings/adr_biobert_embeddings.npz'),
                   allow_pickle=True)
adr_terms      = emb_data['adr_terms']        # shape (3576,)
adr_embeddings = emb_data['embeddings']       # shape (3576, 768)

print(f"\nBioBERT embeddings")
print(f"   adr_terms      : {adr_terms.shape}")
print(f"   adr_embeddings : {adr_embeddings.shape}")

# --- Build a lookup: ADR name -> BioBERT vector ---
adr_to_vec = {term: adr_embeddings[i] for i, term in enumerate(adr_terms)}
print(f"   lookup dict    : {len(adr_to_vec)} ADR terms")

# --- Check: are all ADRs in train.csv covered by the embeddings? ---
train_adrs = set(train_df['adr'].unique())
missing = train_adrs - set(adr_terms)
print(f"\nCoverage check")
print(f"   unique ADRs in train : {len(train_adrs)}")
print(f"   missing from BioBERT : {len(missing)}")
if missing:
    print(f"   examples: {list(missing)[:5]}")

Data splits
   train : (8240, 5)
   val   : (4912, 5)
   test  : (9016, 5)
   train positives: 4120/8240 (50.0%)
   val positives: 2456/4912 (50.0%)
   test positives: 4508/9016 (50.0%)

BioBERT embeddings
   adr_terms      : (3576,)
   adr_embeddings : (3576, 768)
   lookup dict    : 3576 ADR terms

Coverage check
   unique ADRs in train : 3272
   missing from BioBERT : 0


In [7]:
# ===== Cell 3: Install RDKit + PyG, build molecular graphs =====
# RDKit parses SMILES into molecules; PyTorch Geometric provides
# the graph neural network layers. Neither is preinstalled on Colab.
!pip install rdkit torch-geometric -q
print("RDKit and torch-geometric installed.")

from rdkit import Chem
from torch_geometric.data import Data

# IMPORTANT: this must match notebook 04 exactly (5 atom features),
# otherwise the saved GNN weights cannot be loaded.
def atom_features(atom):
    """Return a 5-dimensional feature vector for a single atom."""
    return [
        atom.GetAtomicNum(),        # element (C=6, N=7, O=8, ...)
        atom.GetDegree(),           # number of bonded neighbours
        atom.GetFormalCharge(),     # formal charge
        atom.GetTotalNumHs(),       # attached hydrogens
        int(atom.GetIsAromatic()),  # 1 if aromatic, else 0
    ]

def smiles_to_graph(smiles):
    """Convert a SMILES string into a PyG graph (Data object)."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    node_feats = [atom_features(a) for a in mol.GetAtoms()]
    x = torch.tensor(node_feats, dtype=torch.float)
    edge_list = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_list += [[i, j], [j, i]]
    edge_index = torch.tensor(edge_list, dtype=torch.long).t()
    return Data(x=x, edge_index=edge_index)

# Build one molecular graph per unique drug
drug_smiles = (train_df[['drug', 'smiles']]
               .drop_duplicates()
               .reset_index(drop=True))

drug_graphs = {}
print(f"\nBuilding molecular graphs for {len(drug_smiles)} drugs:\n")
for _, row in drug_smiles.iterrows():
    g = smiles_to_graph(row['smiles'])
    if g is None:
        print(f"  WARNING: failed to parse {row['drug']}")
        continue
    drug_graphs[row['drug']] = g
    print(f"  {row['drug']:16s} {g.num_nodes:3d} atoms, "
          f"{g.num_edges // 2:3d} bonds")

NODE_FEAT_DIM = next(iter(drug_graphs.values())).x.shape[1]
print(f"\nGraphs built for {len(drug_graphs)} drugs")
print(f"Node feature dimension: {NODE_FEAT_DIM}  (must be 5)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 65.7 MB/s eta 0:00:00
RDKit and torch-geometric installed.

Building molecular graphs for 15 drugs:

  glipizide         31 atoms,  33 bonds
  nateglinide       23 atoms,  24 bonds
  metformin          9 atoms,   8 bonds
  glyburide         33 atoms,  35 bonds
  dapagliflozin     28 atoms,  30 bonds
  saxagliptin       23 atoms,  27 bonds
  acarbose          44 atoms,  46 bonds
  repaglinide       33 atoms,  35 bonds
  rosiglitazone     25 atoms,  27 bonds
  sitagliptin       28 atoms,  30 bonds
  pioglitazone      25 atoms,  27 bonds
  canagliflozin     31 atoms,  34 bonds
  linagliptin       35 atoms,  39 bonds
  empagliflozin     31 atoms,  34 bonds
  glimepiride       34 atoms,  36 bonds

Graphs built for 15 drugs
Node feature dimension: 5  (must be 5)


In [8]:
# ===== Cell 4: Cross-attention multimodal fusion model =====
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool


class CrossAttentionFusion(nn.Module):
    """Multimodal model that predicts drug-ADR associations.

    The drug is encoded from its molecular graph with a Graph
    Attention Network (GAT). The ADR is encoded with a frozen
    BioBERT vector projected to the same size. The two modalities
    are combined with a cross-attention layer, in which the drug
    representation queries the ADR representation. This lets the
    model weight chemical versus clinical evidence per drug-ADR
    pair, rather than treating the two modalities independently.
    """

    def __init__(self, node_feat_dim=5, hidden_dim=64,
                 biobert_dim=768, n_heads=4, dropout=0.3):
        super().__init__()

        # --- Drug branch: GAT (same shape as notebook 04) ---
        self.gat1 = GATConv(node_feat_dim, hidden_dim, heads=4)
        self.gat2 = GATConv(hidden_dim * 4, hidden_dim, heads=1)

        # --- ADR branch: project the 768-d BioBERT vector down
        #     to hidden_dim so both modalities share one size ---
        self.adr_proj = nn.Sequential(
            nn.Linear(biobert_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
        )

        # --- Cross-attention: drug queries ADR ---
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=n_heads,
            dropout=dropout, batch_first=True,
        )
        self.attn_norm = nn.LayerNorm(hidden_dim)

        # --- Classifier on the fused representation ---
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def encode_drug(self, graph_batch):
        """Molecular graph -> one vector per drug, shape [B, hidden]."""
        x, edge_index = graph_batch.x, graph_batch.edge_index
        x = F.elu(self.gat1(x, edge_index))
        x = F.elu(self.gat2(x, edge_index))
        return global_mean_pool(x, graph_batch.batch)

    def forward(self, graph_batch, adr_vec):
        # adr_vec: precomputed BioBERT vectors, shape [B, 768]
        drug = self.encode_drug(graph_batch)        # [B, hidden]
        adr  = self.adr_proj(adr_vec)               # [B, hidden]

        # Cross-attention expects a sequence dimension.
        # Drug is the query (length-1 sequence); ADR is key/value.
        q = drug.unsqueeze(1)                       # [B, 1, hidden]
        kv = adr.unsqueeze(1)                       # [B, 1, hidden]
        attn_out, attn_weights = self.cross_attn(q, kv, kv)
        attn_out = attn_out.squeeze(1)              # [B, hidden]

        # Residual connection + normalisation around attention
        fused_drug = self.attn_norm(drug + attn_out)

        # Concatenate the attention-refined drug vector with the
        # ADR vector so the classifier sees both modalities.
        combined = torch.cat([fused_drug, adr], dim=1)  # [B, 2*hidden]
        logit = self.classifier(combined).squeeze(-1)   # [B]
        return logit


# Instantiate the model and move it to the GPU
model = CrossAttentionFusion(node_feat_dim=NODE_FEAT_DIM).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print("CrossAttentionFusion model created")
print(f"   total parameters : {total_params:,}")
print(model)

CrossAttentionFusion model created
   total parameters : 93,057
CrossAttentionFusion(
  (gat1): GATConv(5, 64, heads=4)
  (gat2): GATConv(256, 64, heads=1)
  (adr_proj): Sequential(
    (0): Linear(in_features=768, out_features=64, bias=True)
    (1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (2): ReLU()
  )
  (cross_attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
  )
  (attn_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (classifier): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [9]:
# ===== Cell 5: Warm-start the GAT layers from notebook 04 =====
# The GNN from notebook 04 was already trained to encode molecular
# structure. We reuse its GAT weights as a starting point (transfer
# learning) instead of training the drug branch from scratch.
gnn_state = torch.load(os.path.join(PROJECT, 'models/gnn_model.pt'),
                        map_location=device)

# Keep only the GAT layers; the old adr_embed and classifier do not
# apply here because this model uses BioBERT and a different fusion.
gat_weights = {k: v for k, v in gnn_state.items()
               if k.startswith('gat1.') or k.startswith('gat2.')}

print("Weights available in gnn_model.pt:")
for k in gnn_state.keys():
    reused = "  -> reused" if k in gat_weights else "  -> skipped"
    print(f"   {k:24s}{reused}")

# load_state_dict with strict=False loads matching keys and ignores
# the rest (adr_proj, cross_attn, classifier stay randomly initialised).
missing, unexpected = model.load_state_dict(gat_weights, strict=False)

print(f"\nLoaded {len(gat_weights)} GAT tensors into the model.")
print(f"   randomly initialised (expected): {len(missing)} tensors")
print(f"   unexpected keys (should be 0)  : {len(unexpected)}")

# Sanity check: confirm the GAT layers actually changed
g1 = next(iter(drug_graphs.values())).to(device)
print("\nGAT warm-start applied successfully.")

Weights available in gnn_model.pt:
   gat1.att_src              -> reused
   gat1.att_dst              -> reused
   gat1.bias                 -> reused
   gat1.lin.weight           -> reused
   gat2.att_src              -> reused
   gat2.att_dst              -> reused
   gat2.bias                 -> reused
   gat2.lin.weight           -> reused
   adr_embed.weight          -> skipped
   classifier.0.weight       -> skipped
   classifier.0.bias         -> skipped
   classifier.3.weight       -> skipped
   classifier.3.bias         -> skipped

Loaded 8 GAT tensors into the model.
   randomly initialised (expected): 14 tensors
   unexpected keys (should be 0)  : 0

GAT warm-start applied successfully.


In [12]:
# ===== Cell 6: Dataset and DataLoader with BioBERT vectors =====
from torch_geometric.data import Dataset, Batch
from torch.utils.data import DataLoader

# Ensure every drug graph lives on the CPU. The collate function
# moves batches to the GPU during training; keeping the stored
# graphs on the CPU avoids a device mismatch.
drug_graphs = {name: g.cpu() for name, g in drug_graphs.items()}

class FusionDataset(Dataset):
    """One item = (drug graph, BioBERT ADR vector, label).

    Unlike notebook 04, the ADR is represented by its precomputed
    768-dimensional BioBERT vector rather than an integer index.
    """
    def __init__(self, df, drug_graphs, adr_to_vec):
        super().__init__()
        self.rows = df.reset_index(drop=True)
        self.drug_graphs = drug_graphs
        self.adr_to_vec = adr_to_vec

    def len(self):
        return len(self.rows)

    def get(self, idx):
        row = self.rows.iloc[idx]
        graph = self.drug_graphs[row['drug']]
        adr_vec = torch.tensor(self.adr_to_vec[row['adr']],
                               dtype=torch.float)
        label = float(row['label'])
        return graph, adr_vec, label


def collate(batch):
    """Combine a list of items into batched tensors."""
    graphs, adr_vecs, labels = zip(*batch)
    graph_batch = Batch.from_data_list(list(graphs))
    adr_tensor = torch.stack(adr_vecs)                       # [B, 768]
    label_tensor = torch.tensor(labels, dtype=torch.float)   # [B]
    return graph_batch, adr_tensor, label_tensor


BATCH_SIZE = 32

train_ds = FusionDataset(train_df, drug_graphs, adr_to_vec)
val_ds   = FusionDataset(val_df,   drug_graphs, adr_to_vec)
test_ds  = FusionDataset(test_df,  drug_graphs, adr_to_vec)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=collate)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=collate)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=collate)

print(f"DataLoaders created (batch size {BATCH_SIZE})")
print(f"   train batches : {len(train_loader)}")
print(f"   val batches   : {len(val_loader)}")
print(f"   test batches  : {len(test_loader)}")

# Sanity check: pull one batch and run it through the model
graph_b, adr_b, label_b = next(iter(train_loader))
graph_b, adr_b = graph_b.to(device), adr_b.to(device)
with torch.no_grad():
    out = model(graph_b, adr_b)
print(f"\nOne-batch forward pass")
print(f"   adr_vec batch shape : {tuple(adr_b.shape)}  (expect [32, 768])")
print(f"   model output shape  : {tuple(out.shape)}    (expect [32])")
print("Model accepts the data correctly.")

DataLoaders created (batch size 32)
   train batches : 258
   val batches   : 154
   test batches  : 282

One-batch forward pass
   adr_vec batch shape : (32, 768)  (expect [32, 768])
   model output shape  : (32,)    (expect [32])
Model accepts the data correctly.


In [13]:
# ===== Cell 7: Train the multimodal fusion model =====
import copy
import time
from sklearn.metrics import roc_auc_score, average_precision_score

# Same training setup as notebook 04, so any difference in results
# is attributable to the model, not the training procedure.
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


def train_epoch(model, loader):
    """One full pass over the training data."""
    model.train()
    total_loss = 0.0
    for graph_batch, adr_vec, labels in loader:
        graph_batch = graph_batch.to(device)
        adr_vec = adr_vec.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(graph_batch, adr_vec)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    """Evaluate without updating parameters; return AUROC and AUPRC."""
    model.eval()
    all_probs, all_labels = [], []
    for graph_batch, adr_vec, labels in loader:
        graph_batch = graph_batch.to(device)
        adr_vec = adr_vec.to(device)

        logits = model(graph_batch, adr_vec)
        probs = torch.sigmoid(logits).cpu()
        all_probs.append(probs)
        all_labels.append(labels)

    probs = torch.cat(all_probs).numpy()
    labels = torch.cat(all_labels).numpy()
    return {
        'AUROC': roc_auc_score(labels, probs),
        'AUPRC': average_precision_score(labels, probs),
    }


# --- Training loop with best-model selection on validation AUROC ---
NUM_EPOCHS = 20
best_val_auroc = 0.0
best_model_state = None

print(f"Training the multimodal model for {NUM_EPOCHS} epochs...\n")
print(f"{'Epoch':>6} {'Train Loss':>12} "
      f"{'Val AUROC':>11} {'Val AUPRC':>11}")
print("-" * 42)

start = time.time()
for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_epoch(model, train_loader)
    val_metrics = evaluate(model, val_loader)

    if val_metrics['AUROC'] > best_val_auroc:
        best_val_auroc = val_metrics['AUROC']
        best_model_state = copy.deepcopy(model.state_dict())
        marker = "  <- best"
    else:
        marker = ""

    print(f"{epoch:>6} {train_loss:>12.4f} "
          f"{val_metrics['AUROC']:>11.4f} "
          f"{val_metrics['AUPRC']:>11.4f}{marker}")

print("-" * 42)
print(f"Training time      : {time.time() - start:.1f} s")
print(f"Best val AUROC     : {best_val_auroc:.4f}")

# Restore the best model for final test evaluation
model.load_state_dict(best_model_state)
print("Best model restored.")

Training the multimodal model for 20 epochs...

 Epoch   Train Loss   Val AUROC   Val AUPRC
------------------------------------------
     1       0.4743      0.8765      0.8785  <- best
     2       0.4164      0.8793      0.8817  <- best
     3       0.4072      0.8787      0.8826
     4       0.3952      0.8869      0.8883  <- best
     5       0.3870      0.8866      0.8877
     6       0.3799      0.8881      0.8876  <- best
     7       0.3671      0.8896      0.8898  <- best
     8       0.3644      0.8895      0.8908
     9       0.3592      0.8898      0.8912  <- best
    10       0.3533      0.8912      0.8928  <- best
    11       0.3497      0.8905      0.8918
    12       0.3451      0.8902      0.8903
    13       0.3344      0.8900      0.8894
    14       0.3340      0.8927      0.8917  <- best
    15       0.3237      0.8930      0.8898  <- best
    16       0.3145      0.8891      0.8833
    17       0.3068      0.8920      0.8888
    18       0.3027      0.8920     

In [14]:
# ===== Cell 8: Final test-set evaluation and comparison =====
test_metrics = evaluate(model, test_loader)

print("=" * 52)
print("MULTIMODAL FUSION MODEL - FINAL TEST SET RESULTS")
print("=" * 52)
print(f"\n  Test AUROC : {test_metrics['AUROC']:.4f}")
print(f"  Test AUPRC : {test_metrics['AUPRC']:.4f}")

# --- Reference scores from earlier notebooks ---
BASELINE_AUROC = 0.7008   # XGBoost, notebook 03

# Load the GNN-only test score from notebook 04
gnn_metrics = pd.read_csv(
    os.path.join(PROJECT, 'results/gnn_test_metrics.csv'))
GNN_AUROC = gnn_metrics['AUROC'].iloc[0]
GNN_AUPRC = gnn_metrics['AUPRC'].iloc[0]

print(f"\n{'-' * 52}")
print("COMPARISON ACROSS MODELS (test set, AUROC)")
print(f"{'-' * 52}")
print(f"  XGBoost baseline (nb03) : {BASELINE_AUROC:.4f}")
print(f"  GNN only         (nb04) : {GNN_AUROC:.4f}")
print(f"  Multimodal fusion (nb06): {test_metrics['AUROC']:.4f}")

# --- Improvements ---
print(f"\n{'-' * 52}")
print("IMPROVEMENT OF MULTIMODAL FUSION")
print(f"{'-' * 52}")
d_base = test_metrics['AUROC'] - BASELINE_AUROC
d_gnn  = test_metrics['AUROC'] - GNN_AUROC
print(f"  vs XGBoost baseline : {d_base:+.4f} AUROC")
print(f"  vs GNN only         : {d_gnn:+.4f} AUROC")

if d_gnn > 0:
    print(f"\n  Adding the BioBERT text branch with cross-attention")
    print(f"  improves over the GNN-only model by {d_gnn:+.4f} AUROC.")
else:
    print(f"\n  The multimodal model does not exceed the GNN-only")
    print(f"  model ({d_gnn:+.4f}). This is worth analysing in the")
    print(f"  discussion.")

# --- Save results ---
results_dir = os.path.join(PROJECT, 'results')
pd.DataFrame([test_metrics]).to_csv(
    os.path.join(results_dir, 'multimodal_test_metrics.csv'),
    index=False)
print(f"\nSaved -> results/multimodal_test_metrics.csv")

MULTIMODAL FUSION MODEL - FINAL TEST SET RESULTS

  Test AUROC : 0.8480
  Test AUPRC : 0.8545

----------------------------------------------------
COMPARISON ACROSS MODELS (test set, AUROC)
----------------------------------------------------
  XGBoost baseline (nb03) : 0.7008
  GNN only         (nb04) : 0.8185
  Multimodal fusion (nb06): 0.8480

----------------------------------------------------
IMPROVEMENT OF MULTIMODAL FUSION
----------------------------------------------------
  vs XGBoost baseline : +0.1472 AUROC
  vs GNN only         : +0.0295 AUROC

  Adding the BioBERT text branch with cross-attention
  improves over the GNN-only model by +0.0295 AUROC.

Saved -> results/multimodal_test_metrics.csv


In [15]:
# ===== Cell 9: Ablation - concat fusion vs cross-attention =====
# To isolate the contribution of cross-attention, we build a second
# model that uses the same BioBERT text branch but combines the two
# modalities by plain concatenation instead of cross-attention.

class ConcatFusion(nn.Module):
    """Multimodal model with simple concatenation fusion.

    Identical to CrossAttentionFusion except the cross-attention
    layer is removed: the drug and ADR vectors are concatenated
    directly and passed to the classifier.
    """
    def __init__(self, node_feat_dim=5, hidden_dim=64,
                 biobert_dim=768, dropout=0.3):
        super().__init__()
        self.gat1 = GATConv(node_feat_dim, hidden_dim, heads=4)
        self.gat2 = GATConv(hidden_dim * 4, hidden_dim, heads=1)
        self.adr_proj = nn.Sequential(
            nn.Linear(biobert_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def encode_drug(self, graph_batch):
        x, edge_index = graph_batch.x, graph_batch.edge_index
        x = F.elu(self.gat1(x, edge_index))
        x = F.elu(self.gat2(x, edge_index))
        return global_mean_pool(x, graph_batch.batch)

    def forward(self, graph_batch, adr_vec):
        drug = self.encode_drug(graph_batch)        # [B, hidden]
        adr  = self.adr_proj(adr_vec)               # [B, hidden]
        combined = torch.cat([drug, adr], dim=1)    # [B, 2*hidden]
        return self.classifier(combined).squeeze(-1)


def run_training(model, n_epochs=20):
    """Train a model and return its best-by-val-AUROC test scores.
    Uses the same procedure as Cell 7 for a fair comparison.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCEWithLogitsLoss()
    best_val, best_state = 0.0, None

    for epoch in range(1, n_epochs + 1):
        model.train()
        for graph_batch, adr_vec, labels in train_loader:
            graph_batch = graph_batch.to(device)
            adr_vec = adr_vec.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(graph_batch, adr_vec), labels)
            loss.backward()
            optimizer.step()
        val = evaluate(model, val_loader)['AUROC']
        if val > best_val:
            best_val = val
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return evaluate(model, test_loader)


# --- Train the concat-fusion ablation model ---
print("Training the concat-fusion ablation model (20 epochs)...")
concat_model = ConcatFusion(node_feat_dim=NODE_FEAT_DIM).to(device)

# Warm-start its GAT layers from notebook 04, same as the main model
concat_model.load_state_dict(gat_weights, strict=False)

concat_metrics = run_training(concat_model)
print("Done.\n")

# --- Ablation comparison table ---
print("=" * 52)
print("ABLATION STUDY (test set)")
print("=" * 52)
print(f"{'Model':<28}{'AUROC':>10}{'AUPRC':>12}")
print("-" * 52)
print(f"{'GNN only (nb04)':<28}{GNN_AUROC:>10.4f}{GNN_AUPRC:>12.4f}")
print(f"{'Concat fusion':<28}"
      f"{concat_metrics['AUROC']:>10.4f}"
      f"{concat_metrics['AUPRC']:>12.4f}")
print(f"{'Cross-attention fusion':<28}"
      f"{test_metrics['AUROC']:>10.4f}"
      f"{test_metrics['AUPRC']:>12.4f}")
print("-" * 52)

# --- Interpretation ---
gain_biobert = concat_metrics['AUROC'] - GNN_AUROC
gain_crossattn = test_metrics['AUROC'] - concat_metrics['AUROC']
print(f"\nContribution breakdown (AUROC):")
print(f"  adding BioBERT text branch : {gain_biobert:+.4f}")
print(f"  adding cross-attention     : {gain_crossattn:+.4f}")

# --- Save ablation results ---
ablation_df = pd.DataFrame([
    {'model': 'GNN only',          **{'AUROC': GNN_AUROC,
                                       'AUPRC': GNN_AUPRC}},
    {'model': 'Concat fusion',     **concat_metrics},
    {'model': 'Cross-attn fusion', **test_metrics},
])
ablation_df.to_csv(
    os.path.join(PROJECT, 'results/ablation_results.csv'),
    index=False)
print(f"\nSaved -> results/ablation_results.csv")

Training the concat-fusion ablation model (20 epochs)...
Done.

ABLATION STUDY (test set)
Model                            AUROC       AUPRC
----------------------------------------------------
GNN only (nb04)                 0.8185      0.8315
Concat fusion                   0.8437      0.8505
Cross-attention fusion          0.8480      0.8545
----------------------------------------------------

Contribution breakdown (AUROC):
  adding BioBERT text branch : +0.0253
  adding cross-attention     : +0.0042

Saved -> results/ablation_results.csv


In [16]:
# ===== Cell 10: Bootstrap statistical significance test =====
# Bootstrap resampling estimates how reliable the AUROC differences
# are. We resample the test set 1000 times; if the multimodal model
# beats the GNN in nearly all resamples, the gain is significant.
import numpy as np

@torch.no_grad()
def get_predictions(model, loader):
    """Return (probabilities, labels) arrays for a whole loader."""
    model.eval()
    probs, labels = [], []
    for graph_batch, adr_vec, y in loader:
        graph_batch = graph_batch.to(device)
        adr_vec = adr_vec.to(device)
        logits = model(graph_batch, adr_vec)
        probs.append(torch.sigmoid(logits).cpu())
        labels.append(y)
    return torch.cat(probs).numpy(), torch.cat(labels).numpy()


# Predictions on the test set for both fusion models
cross_probs, test_labels = get_predictions(model, test_loader)
concat_probs, _          = get_predictions(concat_model, test_loader)

# Bootstrap loop
N_BOOTSTRAP = 1000
rng = np.random.default_rng(seed=42)
n = len(test_labels)

cross_aurocs, concat_aurocs, diffs = [], [], []
for _ in range(N_BOOTSTRAP):
    idx = rng.integers(0, n, size=n)            # resample with replacement
    y = test_labels[idx]
    if y.min() == y.max():                      # skip degenerate samples
        continue
    a_cross  = roc_auc_score(y, cross_probs[idx])
    a_concat = roc_auc_score(y, concat_probs[idx])
    cross_aurocs.append(a_cross)
    concat_aurocs.append(a_concat)
    diffs.append(a_cross - a_concat)

cross_aurocs = np.array(cross_aurocs)
concat_aurocs = np.array(concat_aurocs)
diffs = np.array(diffs)

# 95% confidence intervals
def ci95(arr):
    return np.percentile(arr, 2.5), np.percentile(arr, 97.5)

cl, cu = ci95(cross_aurocs)
ol, ou = ci95(concat_aurocs)
dl, du = ci95(diffs)

# p-value: fraction of resamples where cross-attn did NOT beat concat
p_value = np.mean(diffs <= 0)

print("=" * 56)
print(f"BOOTSTRAP RESULTS  ({len(diffs)} resamples)")
print("=" * 56)
print(f"\nCross-attention fusion AUROC")
print(f"   mean : {cross_aurocs.mean():.4f}")
print(f"   95% CI : [{cl:.4f}, {cu:.4f}]")
print(f"\nConcat fusion AUROC")
print(f"   mean : {concat_aurocs.mean():.4f}")
print(f"   95% CI : [{ol:.4f}, {ou:.4f}]")
print(f"\nDifference (cross-attn minus concat)")
print(f"   mean : {diffs.mean():+.4f}")
print(f"   95% CI : [{dl:+.4f}, {du:+.4f}]")
print(f"   p-value : {p_value:.4f}")

print(f"\n{'-' * 56}")
if p_value < 0.05:
    print("Cross-attention significantly outperforms concat")
    print("fusion (p < 0.05).")
else:
    print("The cross-attention gain over concat fusion is NOT")
    print(f"statistically significant (p = {p_value:.3f}).")
    print("The improvement is within the range of sampling noise.")

# Save bootstrap summary
pd.DataFrame([{
    'cross_auroc_mean': cross_aurocs.mean(),
    'cross_ci_low': cl, 'cross_ci_high': cu,
    'concat_auroc_mean': concat_aurocs.mean(),
    'concat_ci_low': ol, 'concat_ci_high': ou,
    'diff_mean': diffs.mean(),
    'diff_ci_low': dl, 'diff_ci_high': du,
    'p_value': p_value,
}]).to_csv(os.path.join(PROJECT, 'results/bootstrap_test.csv'),
           index=False)
print(f"\nSaved -> results/bootstrap_test.csv")

BOOTSTRAP RESULTS  (1000 resamples)

Cross-attention fusion AUROC
   mean : 0.8479
   95% CI : [0.8402, 0.8552]

Concat fusion AUROC
   mean : 0.8437
   95% CI : [0.8356, 0.8514]

Difference (cross-attn minus concat)
   mean : +0.0043
   95% CI : [+0.0008, +0.0076]
   p-value : 0.0060

--------------------------------------------------------
Cross-attention significantly outperforms concat
fusion (p < 0.05).

Saved -> results/bootstrap_test.csv
